In [30]:
# augmentation version 2: here ONLY angry and sad were augmented.
# happy was undersampled to 4000, no new images were generated.
# all three classes are now equal at 4000 images each.

In [31]:
from pathlib import Path
from collections import Counter

from capstone_data.image_loader import load_dataset

data_path = Path.home() / "Downloads" / "cleaned_data"

print("Dataset path:", data_path)
print("Dataset folder exists:", data_path.exists())

Dataset path: /Users/melissatobias/Downloads/cleaned_data
Dataset folder exists: True


In [32]:
from capstone_data.image_loader import load_dataset 
from collections import Counter

data_path = Path.home() / "Downloads" / "cleaned_data"

dataset = load_dataset (data_path)

print (f"Total images: {len (dataset)}")

print("\nImages per class:")
class_counts = Counter (item["class_name"] for item in dataset)

for class_name, count in class_counts. items():
    print (f"{class_name}: {count}")

if class_counts:
    print(f"\nTotal classes found: {len(class_counts)}")
    print(f"Largest class: {max(class_counts, key=class_counts.get)}")
    print(f"Smallest class: {min(class_counts, key=class_counts.get)}")

Total images: 16779

Images per class:
angry: 3089
happy: 6677
sad: 1207
neutral: 5806

Total classes found: 4
Largest class: happy
Smallest class: sad


In [33]:
# splitting 70-15-15

import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

# convert the loader's records into a DataFrame
df = pd.DataFrame(dataset)

# use each record's class name as its stratification label
X = df
y = df["class_name"].str.lower()

# reserve 15% for the final test set
train_val_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=y)

# reserve 15% of the original data for validation.
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_val_df["class_name"].str.lower())

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Training images: 11745
Validation images: 2517
Test images: 2517
Total: 16779


In [34]:

# checking distributions after split
classes = ["angry", "happy", "sad"]

for split_name, split_df in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)]:
    print(f"\n{split_name} distribution:")

    counts = Counter(split_df["class_name"].str.lower())

    for class_name in classes:
        print(f"  {class_name.capitalize():<8}: {counts.get(class_name, 0)}")


Training distribution:
  Angry   : 2163
  Happy   : 4673
  Sad     : 845

Validation distribution:
  Angry   : 463
  Happy   : 1002
  Sad     : 181

Test distribution:
  Angry   : 463
  Happy   : 1002
  Sad     : 181


In [35]:

# export train, validation, and test datasets
for emotion in classes:
    print(f"\n{emotion.upper()}")

# training folder
    class_df = train_3class[
        train_3class["class_name"].str.lower() == emotion
    ].copy()

    # randomly undersample happy
    if emotion == "happy" and len(class_df) > target:
        original_count = len(class_df)
        class_df = class_df.sample(n=target, random_state=42)

        print(f"Happy undersampled: {original_count} -> {target}")

    class_df = class_df.reset_index(drop=True)

    # create train folder
    train_folder = output / emotion / "train"
    train_folder.mkdir(parents=True, exist_ok=True)

    original_images = []

    # save original training images
    for index, row in class_df.iterrows():

        with Image.open(row["path"]) as img:
            gray_img = img.convert("L").copy()

        original_images.append(gray_img)
        gray_img.save(train_folder / f"original_{index:05d}.jpg", format="JPEG", quality=95)

    # augment training images to reach target
    additional_needed = max(0, target - len(original_images))

    for index in range(additional_needed):
        source = random.choice(original_images)
        augmented = augment_grayscale(source)
        augmented.save(train_folder / f"augmented_{index:05d}.jpg", format="JPEG", quality=95)

    print(
        f"Train: {len(original_images)} original + "
        f"{additional_needed} augmented = "
        f"{len(original_images) + additional_needed}")

# validation and test folders
    for split_name, split_df in [
        ("validation", val_df),
        ("test", test_df)]:

        # select images belonging to this class
        split_class_df = split_df[
            split_df["class_name"].str.lower() == emotion
        ].reset_index(drop=True)

        # create output folder
        split_folder = output / emotion / split_name
        split_folder.mkdir(parents=True, exist_ok=True)

        # save original images without augmentation
        for index, row in split_class_df.iterrows():
            with Image.open(row["path"]) as img:
                gray_img = img.convert("L").copy()

            gray_img.save(split_folder / f"original_{index:05d}.jpg", format="JPEG", quality=95)
        print(f"{split_name.capitalize()}: {len(split_class_df)} images")

print("Output location:", output)



ANGRY
Train: 2163 original + 1837 augmented = 4000
Validation: 463 images
Test: 463 images

HAPPY
Happy undersampled: 4673 -> 4000
Train: 4000 original + 0 augmented = 4000
Validation: 1002 images
Test: 1002 images

SAD
Train: 845 original + 3155 augmented = 4000
Validation: 181 images
Test: 181 images
Output location: /Users/melissatobias/Downloads/Capstone-DATA4381/augmentation/aug_data_v2


In [36]:

def augment_grayscale(image):
    image = image.convert("L")
    original_size = image.size

    # horizontal flip (mirrored)
    if random.random() < 0.5:
        image = ImageOps.mirror(image)

    # small rotation (random left or right)
    if random.random() < 0.7:
        angle = random.uniform(-13, 13)

        image = image.rotate(
            angle,
            resample=Image.Resampling.BICUBIC,
            expand=True,
            fillcolor=0)

    # brightness change
    if random.random() < 0.5:
        image = ImageEnhance.Brightness(image).enhance(
            random.uniform(0.9, 1.2))

    return image.convert("L")


In [37]:

# checking final dataset
print("Final Dataset Verification:")
for emotion in classes:
    print(f"\n{emotion.upper()}")

    for split_name in ["train", "validation", "test"]:

        folder = output / emotion / split_name

        files = [
            p for p in folder.iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]

        print(f"{split_name.capitalize():<12}: {len(files)} images")

        # training set should have 4000 images
        if split_name == "train":
            assert len(files) == target, (f"{emotion} training set should have {target} images")


Final Dataset Verification:

ANGRY
Train       : 4000 images
Validation  : 463 images
Test        : 463 images

HAPPY
Train       : 4000 images
Validation  : 1002 images
Test        : 1002 images

SAD
Train       : 4000 images
Validation  : 181 images
Test        : 181 images
